In [ ]:
# @title 📀 Mount Google Drive
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")
%cd /content/drive/MyDrive/colab/thesis/rrs
CWD = Path.cwd()

In [ ]:
# @title 🧩 Package Installation and Imports
%%capture
!pip install uv
!uv pip install evaluate rouge-score bert-score
!uv pip install git+https://github.com/google-research/bleurt.git

# !uv pip install -q gdown
# !gdown "1pjd8TxqGeYVdCWZHdXow8pDXpwRebwFY" -O "./evaluation/"
# !unzip ./evaluation/ClinicalBLEURT.zip -d ./evaluation/

import evaluate
import numpy as np
import pandas as pd
from bert_score import BERTScorer
from bleurt.score import BleurtScorer

In [ ]:
# @title 📊 Test Dataset
df1 = pd.read_json("data/test-gen3.jsonl", lines=True)
df2 = pd.read_json("data/test-gen3-pegasus.jsonl", lines=True)

In [ ]:
df1.drop(["qwen3_gen", "qwen3x_gen"], axis=1, inplace=True)

In [ ]:
df1.rename(columns={"biobart_gen": "qwen_gen"}, inplace=True)

In [ ]:
df2.drop(["qwen3_gen", "qwen3x_gen"], axis=1, inplace=True)
df2.rename(columns={"pegasus_gen": "qwen_gen"}, inplace=True)

In [ ]:
import pandas as pd
import numpy as np

# Reproducible split
rng = np.random.default_rng(123)

col = "qwen_gen"
df3 = df1.copy()
flip = rng.choice(df1.index, size=len(df1) // 2, replace=False)
df3.loc[flip, col] = df2.loc[flip, col]

In [ ]:
df3.to_json("data/test-gen3-qwen.jsonl", orient="records", lines=True)

In [ ]:
# @title 📊 Test Dataset
test_df = pd.read_json("data/test-gen3-qwen.jsonl", lines=True)
test_df.info()

In [ ]:
# @title 🚀🚀🚀 Metrics
from types import SimpleNamespace

metric = SimpleNamespace()
metric.rouge = evaluate.load("rouge")
metric.bertscore = BERTScorer(model_type="allenai/scibert_scivocab_cased")
metric.bleurtscore = BleurtScorer(checkpoint="evaluation/ClinicalBLEURT")


def compute_rouge(refs, preds):
    rouge = metric.rouge
    return rouge.compute(predictions=preds, references=refs)


def compute_bertscore(refs, preds):
    bertscore = metric.bertscore
    bertscore._tokenizer.model_max_length = 512
    P, R, F = bertscore.score(preds, refs)
    return {
        "bertscore_pr": P.mean().item(),
        "bertscore_re": R.mean().item(),
        "bertscore_f1": F.mean().item(),
    }


def compute_bleurtscore(refs, preds):
    bleurtscore = metric.bleurtscore
    result = bleurtscore.score(references=refs, candidates=preds)
    return { "clinicalbleurt": np.mean(result) }


def compute_metrics_all(refs, preds):
    return {
        **compute_rouge(refs, preds),
        **compute_bertscore(refs, preds),
        **compute_bleurtscore(refs, preds),
    }

In [ ]:
# @title BioBART
from tabulate import tabulate

# results_1 = compute_metrics_all(
#     test_df["impression"].to_list(),
#     test_df["biobart_gen"].to_list()
# )

results_2 = compute_metrics_all(
    test_df["impression"].to_list(),
    test_df["qwen_gen"].to_list()
)

# results_3 = compute_metrics_all(
#     test_df["impression"].to_list(),
#     test_df["qwen3x_gen"].to_list()
# )

headers = ["METRIC", "SCORE"]
# print(tabulate(results_1.items(), headers, tablefmt="simple"))
print(tabulate(results_2.items(), headers, tablefmt="simple"))
# print(tabulate(results_3.items(), headers, tablefmt="simple"))

In [ ]:
# @title Pegasus
from tabulate import tabulate

results_1 = compute_metrics_all(
    test_df["impression"].to_list(),
    test_df["pegasus_gen"].to_list()
)

results_2 = compute_metrics_all(
    test_df["impression"].to_list(),
    test_df["qwen3_gen"].to_list()
)

results_3 = compute_metrics_all(
    test_df["impression"].to_list(),
    test_df["qwen3x_gen"].to_list()
)

headers = ["METRIC", "SCORE"]
print(tabulate(results_1.items(), headers, tablefmt="simple"))
print(tabulate(results_2.items(), headers, tablefmt="simple"))
print(tabulate(results_3.items(), headers, tablefmt="simple"))